# API Playground

Run a cell by clicking it and pressing **Shift + Enter**. Change things and run them again.

## 0. Setup
`requests` is the Python library for making HTTP requests. Colab already has it installed.

In [ ]:
import requests

## 1. First request
Get a random joke.

In [ ]:
response = requests.get("https://official-joke-api.appspot.com/random_joke")

print(response.status_code)   # 200 means it worked
print(response.json())        # the data, as a Python dict

Pull out the fields we want:

In [ ]:
joke = response.json()
print(joke["setup"])
print(joke["punchline"])

## 2. Random dog photo
This API returns a link to an image, which we can display in the notebook.

In [ ]:
from IPython.display import Image

dog = requests.get("https://dog.ceo/api/breeds/image/random").json()
print(dog)
Image(url=dog["message"], width=300)

## 3. Pokemon
Change `name` to another Pokemon (lowercase) and run it again.

In [ ]:
name = "pikachu"

poke = requests.get(f"https://pokeapi.co/api/v2/pokemon/{name}").json()

print("Name:  ", poke["name"])
print("Height:", poke["height"])
print("Weight:", poke["weight"])
print("Types: ", [t["type"]["name"] for t in poke["types"]])
Image(url=poke["sprites"]["front_default"])

## 4. Weather (query parameters)
Options go in `params`, and `requests` adds them to the URL.

Try your own city's latitude and longitude.

In [ ]:
params = {
    "latitude": 29.65,     # Gainesville, FL
    "longitude": -82.32,
    "current": "temperature_2m,wind_speed_10m",
}
r = requests.get("https://api.open-meteo.com/v1/forecast", params=params, timeout=10)

print("URL:", r.url)
weather = r.json()["current"]
print("Temperature:", weather["temperature_2m"], "°C")
print("Wind:", weather["wind_speed_10m"], "km/h")

## 5. POST request
So far we've only read data with GET. POST sends data. JSONPlaceholder is a fake API for practice, it doesn't actually save anything.

In [ ]:
new_post = {"title": "Hello", "body": "My first POST request", "userId": 1}

r = requests.post("https://jsonplaceholder.typicode.com/posts", json=new_post, timeout=10)
print(r.status_code)   # 201 = Created
print(r.json())

## 6. Errors
What happens if we ask for a Pokemon that doesn't exist?

In [ ]:
r = requests.get("https://pokeapi.co/api/v2/pokemon/notarealpokemon", timeout=10)

if r.status_code == 200:
    print(r.json()["name"])
else:
    print("Request failed with status", r.status_code)

## Exercise
Pick one:
- Get a cat fact from `https://catfact.ninja/fact` and print only the fact
- Guess an age from a name: `https://api.agify.io?name=aryan`
- Get a photo of a specific breed: `https://dog.ceo/api/breed/husky/images/random`

Print `r.json()` first to see what the data looks like.

In [ ]:
# your code here

---
# Part 2: Calling an AI model

We'll use Google's Gemini API, which has a free tier.

Setup:
1. Go to [aistudio.google.com](https://aistudio.google.com), click **Get API key**, then **Create API key**
2. In Colab, open the **Secrets** panel (key icon in the left sidebar)
3. Add a secret named `GEMINI_API_KEY`, paste your key, and turn on notebook access

In [ ]:
from google.colab import userdata

API_KEY = userdata.get("GEMINI_API_KEY")
MODEL = "gemini-2.5-flash"   # if this errors, check ai.google.dev for current model names

def ask_ai(prompt):
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"
    headers = {"x-goog-api-key": API_KEY}
    body = {"contents": [{"parts": [{"text": prompt}]}]}

    r = requests.post(url, headers=headers, json=body, timeout=60)
    if r.status_code != 200:
        return f"Error {r.status_code}: {r.text}"
    return r.json()["candidates"][0]["content"]["parts"][0]["text"]

print(ask_ai("Explain what an API is in two sentences."))

Change the prompt and see what you get.

In [ ]:
print(ask_ai("Explain HTTP status codes in 3 bullet points."))

## 7. Combining APIs
Get data from one API and pass it to the model.

In [ ]:
name = "gengar"

poke = requests.get(f"https://pokeapi.co/api/v2/pokemon/{name}").json()
types = [t["type"]["name"] for t in poke["types"]]

prompt = f"Write a short 4-line poem about {name}, a {'/'.join(types)} type Pokemon."
print(ask_ai(prompt))
Image(url=poke["sprites"]["front_default"])

## Final exercise
Use the weather API from section 4 together with `ask_ai` to write a haiku about the current weather in your city.

In [ ]:
# your code here

---
More free APIs: [github.com/public-apis/public-apis](https://github.com/public-apis/public-apis)